# FeINFN - CAVE x4 Fusion Benchmark
Paper: Feature-Enhanced Invertible Fusion Network for Hyperspectral Image Fusion (NeurIPS 2024)
Reported PSNR: 52.47 dB (SOTA)

In [ ]:
import subprocess, glob, os, shutil, sys

# Install correct torch for P100 (sm_60)
try:
    import torch
    cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)
    if cap[0] < 7:
        print(f'P100 detected (sm_{cap[0]}{cap[1]}), need torch 2.5.1+cu121...')
        raise RuntimeError('wrong torch')
    print(f'torch {torch.__version__} OK for sm_{cap[0]}{cap[1]}')
except Exception:
    whls = []
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.whl'):
                whls.append(os.path.join(root, f))
    whls.sort()
    print(f'Found wheels: {whls}')
    if not whls:
        raise RuntimeError('No torch wheels found')
    for w in whls:
        base = os.path.basename(w)
        fixed = base.replace('cu121-cp312', '+cu121-cp312')
        dst = os.path.join('/tmp', fixed)
        shutil.copy2(w, dst)
        print(f'Installing {dst}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--no-deps', dst])
    print('torch install done')
    import torch
print(f'torch {torch.__version__}')
assert torch.cuda.is_available(), 'CUDA not available'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} {p.total_memory/2**30:.1f}GB sm_{p.major}{p.minor}')

In [ ]:
!pip install -q scipy scikit-image einops

In [ ]:
import os, sys, shutil

# Find training script
SCRIPT = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'train_feinfn.py':
            SCRIPT = os.path.join(root, f)
            break
    if SCRIPT:
        break

assert SCRIPT, 'train_feinfn.py not found!'
print(f'Script: {SCRIPT}')
shutil.copy(SCRIPT, '/kaggle/working/train_feinfn.py')
print('Copied to /kaggle/working/')

In [ ]:
%cd /kaggle/working
!python train_feinfn.py 2>&1